# Tools and Functions

In [2]:
// IMPORTS

#include <iostream> 
#include <random>

In [3]:
// VALUE PRINT

template <typename T>
void vprint(const T& x) {
    std::cout << x << std::endl;
}

In [4]:
// ARRAY PRINT

template <typename T>
void aprint(const T* array, size_t size) {
    for (size_t i = 0; i < size; ++i) {
        std::cout << array[i] << " ";
    }
    std::cout << std::endl; 
}

In [5]:
// MATRIX PRINT (ONLY FOR INTEGER VALUES)

void mprint(int** matrix, size_t rows, size_t cols) {
    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j) {
            std::cout << matrix[i][j] << " ";
        }
        std::cout << std::endl;
    }
    std::cout << std::endl;
}

In [6]:
// RANDOM SETUP
// from 1 to 9

std::random_device rd; // Obtain a random number from hardware
std::mt19937 gen(rd()); // Seed the generator
std::uniform_int_distribution<> int_dis(1, 9);

In [7]:
// RANDOM GEN

int randInt(){
    return int_dis(gen);
}

# Cache Basics

Get some information about the CPU on the current system (in that case from the codespace).

In [8]:
! lscpu

Architecture:                       x86_64
CPU op-mode(s):                     32-bit, 64-bit
Byte Order:                         Little Endian
Address sizes:                      48 bits physical, 48 bits virtual
CPU(s):                             2
On-line CPU(s) list:                0,1
Thread(s) per core:                 2
Core(s) per socket:                 1
Socket(s):                          1
NUMA node(s):                       1
Vendor ID:                          AuthenticAMD
CPU family:                         25
Model:                              1
Model name:                         AMD EPYC 7763 64-Core Processor
Stepping:                           1
CPU MHz:                            3241.163
BogoMIPS:                           4890.85
Virtualization:                     AMD-V
Hypervisor vendor:                  Microsoft
Virtualization type:                full
L1d cache:                          32 KiB
L1i cache:                          32 KiB
L2 cache:           

Lets look at the cache by its own.

In [9]:
! lscpu | grep -i "cache"

L1d cache:                          32 KiB
L1i cache:                          32 KiB
L2 cache:                           512 KiB
L3 cache:                           32 MiB


We have 2 CPUs available with 32 KiB of L1 cache (sometimes also known as level 1 cache).

The L1 cache is a type of CPU cache that is closest to the processor cores. It is typically split into two distinct caches: L1i (L1 instruction cache) and L1d (L1 data cache). 

Purpose:

- L1i Cache (Instruction Cache):
    - Stores instructions that the CPU needs to execute.
    - Optimizes the fetching of instructions from memory to the CPU.
    - Improves the performance of the instruction pipeline by reducing the time it takes to fetch instructions.

- L1d Cache (Data Cache):
    - Stores data that the CPU needs to process.
    - Optimizes the fetching of data from memory to the CPU.
    - Enhances the performance of data retrieval operations.The L1 cache, or Level 1 cache, is a type of CPU cache that is closest to the processor cores. It is typically split into two distinct caches: L1i (L1 instruction cache) and L1d (L1 data cache). Here are the differences between them:

We also have 512 KiB of L2 and 32 MiB of L3 cache.

Let look at the cache line size.

In [10]:
! getconf LEVEL1_DCACHE_LINESIZE

64


In [11]:
! getconf LEVEL1_ICACHE_LINESIZE

64


In [12]:
! getconf LEVEL2_CACHE_LINESIZE

64


In [13]:
! getconf LEVEL3_CACHE_LINESIZE

64


The cache line size is 64 bytes.

This means, with a L1 cache size of 32 KiB and a cache line size of 64 bytes we have 500 cache lines.
When storing loading data in the L1 cache, we load a whole cache line, so in that case 64 bytes per load.

# Cpp Basics

Get the size of an integer in byte.

In [14]:
vprint(sizeof(int));

4


This means that we use 4 bytes (**32 bits**) per integer, so $−2^{31}$ to $231^{−1231}−1$ for signed integers.

## Vectors / Arrays with malloc

void* malloc(size_t size);

### How malloc Works

- Requesting Memory: When malloc is called, it requests a block of memory from the heap (**RAM**) of the specified size.
- Allocating Memory: The heap manager in the runtime library tries to find a suitable **contiguous** block of free memory that meets the requested size. If it finds such a block, it marks it as used and returns a **pointer** to the beginning of the block.
- Return Value: If successful, malloc returns a void* pointer to the beginning of the allocated memory block. If it fails (e.g., if there is not enough memory available), it returns NULL.

Lets make an array with 10 integer values.
(Note: We have to keep track of the size and / or type of the array to iterate over it.)

In [15]:
int* arr = (int*)malloc(10 * sizeof(int));

We allocate 10 * 4 bytes with malloc and cast the void pointer to an integer pointer that points to the first element.
This integer pointer must be stored in an integer pointer variable, in that case "arr". 

Lets set the value at positon [0] to 1.

In [16]:
arr[0] = 1;

And print it.

In [17]:
vprint(arr[0]);

1


Lets quickly write a function to fill the array with random integer values.
(The array must be an integer array.)

In [18]:
void fillRandom(int* arr, size_t size) {

    for(int i = 0; i < size; i++){
        arr[i] = randInt();
    }

}

And a function that to fill the array with zeros.

In [19]:
void fillZeros(int* arr, size_t size) {

    for(int i = 0; i < size; i++){
        arr[i] = 0;
    }

}

Lets fill the array with values.

In [20]:
fillRandom(arr, 10);

And print it.

In [21]:
aprint(arr, 10);

9 9 5 1 3 5 6 8 7 7 


## Matrices

We want to create a matrix of size n x m or rows x cols.

In [25]:
size_t rows = 4;
size_t cols = 4; 

A matrix is a set of pointers that are pointing to set of values (vecotrs) each.
Thats why we need to create a set of integer pointers first. 

In [26]:
int** matrix = (int**)malloc(rows * sizeof(int*));

Lets fill our now pointers by allocating the space needed.

In [27]:
for (int i = 0; i < rows; ++i) {
    
    matrix[i] = (int*)malloc(cols * sizeof(int));

}

To fill it, we must create a new function.

(Note: We could use the fillRandom function too.)

In [28]:
void fillMatrixRandom(int** matrix, size_t rows, size_t cols){

    for(int i = 0; i < rows; i++){
        
        for(int j = 0; j < cols; j++){
            
            matrix[i][j] = randInt();

        }
    
    }

}

And fill our matrix.

In [29]:
fillMatrixRandom(matrix, rows, cols);

Now we can print it.

In [30]:
mprint(matrix, rows, cols);

5 2 7 8 
5 7 1 7 
5 1 9 1 
4 2 9 2 



Lets clean everything up and make some functions. 

In [31]:
int* createVector(size_t size){
    int* v = (int*)malloc(size * sizeof(int));
    fillRandom(v, size);
    return v;
}

In [32]:
int* createZeroVector(size_t size){
    int* v = (int*)malloc(size * sizeof(int));
    fillZeros(v, size);
    return v;
}

In [33]:
int** createMatrix(size_t rows, size_t cols){
    int** m = (int**)malloc(rows * sizeof(int*));
    for (size_t i = 0; i < rows; i++) {
        m[i] = createVector(cols);
    }

    return m;
}

In [34]:
int** createZeroMatrix(size_t rows, size_t cols){
    int** m = (int**)malloc(rows * sizeof(int*));
    for (size_t i = 0; i < rows; i++) {
        m[i] = createZeroVector(cols);
    }

    return m;
}

And test the functions.

In [43]:
size_t size = 4;

int** m1 = createMatrix(size, size);
int** m2 = createZeroMatrix(size, size);

mprint(m1, size, size);
mprint(m2, size, size);

5 6 8 5 
9 2 5 4 
7 3 6 4 
5 5 9 8 

0 0 0 0 
0 0 0 0 
0 0 0 0 
0 0 0 0 



## Transpose

Lastly, we write a function to transpose a matrix.

In [37]:
int** transpose(int** in, size_t rows, size_t cols){

    int** out = createMatrix(cols, rows);

    for(size_t row = 0; row < rows; row++){

        for(size_t col = 0; col < cols; col++){
    
            out[col][row] = in[row][col];
            
        }
    }
    
    return out;

}

And test the function.

In [38]:
size_t rows = 5;
size_t cols = 3;

int** in = createMatrix(rows, cols);
mprint(in, rows, cols);

int** trans = transpose(in, rows, cols);

// Note the change from rows / cols to cols / rows !!!
mprint(trans, cols, rows);

7 5 5 
4 8 7 
5 3 8 
2 3 4 
1 6 9 

7 4 5 2 1 
5 8 3 3 6 
5 7 8 4 9 



# Matrix Operations

Lets look in some ways to perform the dot product.

## Dot product of 2 arrays

In [44]:
int dot(int* arr1, int* arr2, size_t size){
    int sum = 0;
    for(int i = 0; i < size; i++){
        sum += arr1[i] * arr2[i];
    }
    return sum;
}

We know, that we have 64 bytes cache line size and 4 bytes per integer.
That means, that we can potentially load 16 integer in one cache line.
And that means, that we potentially only have to load 2 times from RAM to perform a 16 x 16 dot product.  

Note:
This is a simplified view, as real-world scenarios involve considerations like alignment and data structures etc.

In [45]:
int size = 16;

int* arr1 = createVector(size);
int* arr2 = createVector(size);

aprint(arr1, size);
aprint(arr2, size);

7 9 3 8 6 3 1 1 3 9 6 9 7 8 9 9 
5 7 6 1 1 6 4 4 7 5 8 2 3 2 7 4 


In [46]:
vprint(dot(arr1, arr2, size))

424


## Standart dot product for matrices.

(Note: For now, we assume that we have quadratic matrices, row = cols.)

In [ ]:
void mDotQuad(int** mat1, int** mat2, int** out, size_t size){

    for(int row = 0; row < size; row++){

        for(int col = 0; col < size; col++){

            for(int k = 0; k < size; k++){
                
                out[row][col] += mat1[row][k] * mat2[k][col];
            }
        }
    }
}

And test it.

In [47]:
size_t size = 16;

int** m1 = createMatrix(size, size);
int** m2 = createMatrix(size, size);
int** out = createZeroMatrix(size, size);

mprint(m1, size, size);
mprint(m2, size, size);

mDotQuad(m1, m2, out, size);

mprint(out, size, size);

9 8 9 6 3 9 6 7 2 8 4 2 1 8 7 9 
9 9 9 9 1 1 7 8 7 8 8 6 1 1 7 4 
1 5 2 8 4 1 1 3 5 9 3 5 3 7 7 7 
5 1 6 9 8 7 4 7 1 3 8 2 7 3 8 8 
3 9 9 2 9 3 2 3 1 2 9 2 5 9 1 9 
9 2 1 1 7 6 5 1 7 2 3 5 3 6 9 4 
5 3 7 1 9 2 5 3 8 7 6 5 4 7 4 3 
6 4 9 1 7 1 6 2 9 4 6 9 2 7 3 6 
2 6 1 5 5 9 7 5 3 9 8 3 7 6 8 2 
7 5 1 6 7 3 5 7 1 2 7 8 4 1 5 1 
9 3 9 2 2 8 3 9 7 2 2 1 4 4 1 9 
4 5 3 7 9 6 5 2 4 8 1 6 2 7 6 3 
8 7 3 7 4 7 3 9 6 1 9 8 6 6 3 3 
5 6 1 4 4 3 6 3 6 2 4 6 9 8 1 8 
7 3 5 4 3 7 3 6 8 3 9 4 5 4 3 9 
9 2 6 2 8 8 4 8 2 2 3 8 7 5 4 5 

2 9 1 6 2 7 8 7 7 7 2 7 1 1 5 5 
8 9 2 3 5 8 2 1 5 6 6 8 6 8 2 2 
8 6 1 3 2 9 7 1 7 8 1 6 8 2 5 8 
8 7 2 2 7 5 8 8 1 6 3 1 2 6 7 2 
7 7 1 8 8 8 3 7 2 1 4 9 7 4 2 8 
5 1 6 5 9 1 1 6 5 8 2 9 1 6 6 7 
6 1 6 5 5 9 9 9 5 4 2 2 7 1 8 9 
8 9 2 4 7 4 2 1 9 4 3 6 1 1 8 9 
5 6 5 7 6 5 1 7 9 5 9 6 2 8 7 7 
9 1 4 1 5 1 5 9 8 4 9 3 6 2 2 3 
4 9 7 9 5 3 4 9 1 2 1 8 5 5 9 4 
8 2 8 8 4 5 9 5 5 8 4 6 8 7 5 3 
6 2 3 9 5 9 9 4 8 2 9 6 5 3 8 5 
7 6 6 4 3 2 5 9 4 4 5 8 3 2 8 6 
5 3 5 8 7

# Cleanup

At the end, we must free the allocated memory space.

In [ ]:
void freeM(int** matrix, int n) {
    for (int i = 0; i < n; ++i) {
        free(matrix[i]); 
    }
    free(matrix); 
}

In [ ]:
// free(arr);
// free(arr1);
// free(arr2);
// freeM(matrix, rows);
// freeM(m1, size);
// freeM(m2, size);
// freeM(out, size);